In [ ]:
import pandas as pd

csv = pd.read_csv(r"AC_cast硬度数据集_483.csv")
print(csv.columns)

Index(['Unnamed: 0', 'Composition', 'VEC', 'Electronegativity_Difference',
       'Atomic_Radius_Diff', 'Mixing_Enthalpy', 'Mixing_Entropy', 'E/A', 'EWF',
       'Mod_Mismatch', 'DeltaG', 'ShearModG', 'Tm', 'Ec', 'Cond__AC',
       'Cond__AM', 'Cond__HM', 'Cond__PM', 'Cond__WR', 'Li', 'Mg', 'Al', 'Si',
       'Sc', 'Ti', 'V', 'Cr', 'Mn', 'Fe', 'Ni', 'Co', 'Cu', 'Zn', 'Zr', 'Nb',
       'Mo', 'Sn', 'Hf', 'Ta', 'W', 'HV'],
      dtype='object')


In [ ]:
"""
高熵合金知识图谱系统 - 纯文本报告版
专注于详细数值分析，不生成图表
"""

import torch
import numpy as np
import pandas as pd
from transformers import AutoTokenizer, AutoModel
from sklearn.metrics.pairwise import cosine_similarity
from datetime import datetime
import warnings

warnings.filterwarnings('ignore')

# ============================================================================
# 配置部分
# ============================================================================

MODEL_PATH = r"model_results_save\final_model"

# 元素分类（基于原子半径、电负性等）
ELEMENT_CATEGORIES = {
    'light_elements': ['Mg', 'Al', 'Si', 'Ti', 'V', 'Cr', 'Mn'],
    'transition_metals': ['Fe', 'Co', 'Ni', 'Cu', 'Zn'],
    'refractory_elements': ['Zr', 'Nb', 'Mo', 'Hf', 'Ta', 'W'],
    'rare_elements': ['Sc', 'Sn']
}

# 元素特性
ELEMENT_PROPERTIES = {
    'Mg': {'atomic_radius': 1.60, 'electronegativity': 1.31, 'valence': 2},
    'Al': {'atomic_radius': 1.43, 'electronegativity': 1.61, 'valence': 3},
    'Si': {'atomic_radius': 1.18, 'electronegativity': 1.90, 'valence': 4},
    'Sc': {'atomic_radius': 1.62, 'electronegativity': 1.36, 'valence': 3},
    'Ti': {'atomic_radius': 1.47, 'electronegativity': 1.54, 'valence': 4},
    'V': {'atomic_radius': 1.34, 'electronegativity': 1.63, 'valence': 5},
    'Cr': {'atomic_radius': 1.28, 'electronegativity': 1.66, 'valence': 6},
    'Mn': {'atomic_radius': 1.27, 'electronegativity': 1.55, 'valence': 7},
    'Fe': {'atomic_radius': 1.26, 'electronegativity': 1.83, 'valence': 8},
    'Ni': {'atomic_radius': 1.24, 'electronegativity': 1.91, 'valence': 10},
    'Co': {'atomic_radius': 1.25, 'electronegativity': 1.88, 'valence': 9},
    'Cu': {'atomic_radius': 1.28, 'electronegativity': 1.90, 'valence': 11},
    'Zn': {'atomic_radius': 1.34, 'electronegativity': 1.65, 'valence': 12},
    'Zr': {'atomic_radius': 1.60, 'electronegativity': 1.33, 'valence': 4},
    'Nb': {'atomic_radius': 1.46, 'electronegativity': 1.60, 'valence': 5},
    'Mo': {'atomic_radius': 1.39, 'electronegativity': 2.16, 'valence': 6},
    'Sn': {'atomic_radius': 1.41, 'electronegativity': 1.96, 'valence': 4},
    'Hf': {'atomic_radius': 1.59, 'electronegativity': 1.30, 'valence': 4},
    'Ta': {'atomic_radius': 1.46, 'electronegativity': 1.50, 'valence': 5},
    'W': {'atomic_radius': 1.39, 'electronegativity': 2.36, 'valence': 6}
}

# 增强的模板系统
TEMPLATES = {
    'element': [
        "The {element} element in high entropy alloys",
        "Effect of {element} addition on HEA properties",
        "Role of {element} in alloy strengthening mechanisms",
        "Contribution of {element} to mechanical performance",
        "Influence of {element} on hardness and strength"
    ],
    'mechanism': [
        "{mechanism} strengthening effect in HEAs",
        "Importance of {mechanism} for alloy hardening",
        "{mechanism} contributes to improved properties",
        "The {mechanism} mechanism in multi-principal element alloys",
        "Strengthening by {mechanism} in complex alloys"
    ],
    'property': [
        "{property} of high entropy alloys",
        "High {property} achieved through alloy design",
        "{property} improvement in advanced materials",
        "Measurement and optimization of {property}",
        "Superior {property} in multicomponent systems"
    ]
}

# ============================================================================
# 核心类定义
# ============================================================================

class HEAKnowledgeExtractor:
    """高熵合金知识提取器"""
    
    def __init__(self, model_path):
        print("=" * 80)
        print("高熵合金知识图谱系统 - 纯文本报告版")
        print("=" * 80)
        
        print("[1/4] 加载预训练模型...")
        self.tokenizer = AutoTokenizer.from_pretrained(model_path)
        self.model = AutoModel.from_pretrained(model_path)
        self.model.eval()
        
        # 初始化数据结构
        self.element_embeddings = {}
        self.mechanism_embeddings = {}
        self.property_embeddings = {}
        
    def get_enhanced_embedding(self, entity, entity_type):
        """获取增强的嵌入向量"""
        if entity_type == 'element':
            templates = TEMPLATES['element']
            text_entity = entity
        elif entity_type == 'mechanism':
            templates = TEMPLATES['mechanism']
            text_entity = entity
        else:  # property
            templates = TEMPLATES['property']
            text_entity = entity
        
        embeddings = []
        for template in templates:
            text = template.format(**{entity_type: text_entity})
            inputs = self.tokenizer(text, return_tensors='pt', 
                                   padding=True, truncation=True, max_length=128)
            with torch.no_grad():
                outputs = self.model(**inputs)
            mean_embedding = outputs.last_hidden_state[0].mean(dim=0).numpy()
            embeddings.append(mean_embedding)
        
        return np.mean(embeddings, axis=0)
    
    def extract_all_embeddings(self):
        """提取所有实体的嵌入向量"""
        print("\n[2/4] 提取实体嵌入向量...")
        
        # 提取元素嵌入
        print("  提取元素嵌入...")
        elements = list(ELEMENT_PROPERTIES.keys())
        for i, element in enumerate(elements, 1):
            self.element_embeddings[element] = self.get_enhanced_embedding(element, 'element')
            if i % 5 == 0 or i == len(elements):
                print(f"    进度: {i}/{len(elements)}")
        
        # 定义机制
        mechanisms = [
            'solid solution strengthening',
            'lattice distortion',
            'precipitation strengthening',
            'grain refinement',
            'phase transformation',
            'dislocation strengthening',
            'twining induced plasticity',
            'transformation induced plasticity'
        ]
        
        # 提取机制嵌入
        print("\n  提取强化机制嵌入...")
        for i, mechanism in enumerate(mechanisms, 1):
            self.mechanism_embeddings[mechanism] = self.get_enhanced_embedding(mechanism, 'mechanism')
            if i % 2 == 0 or i == len(mechanisms):
                print(f"    进度: {i}/{len(mechanisms)}")
        
        # 定义性能
        properties = ['hardness', 'strength', 'ductility', 'toughness', 'wear resistance', 
                      'corrosion resistance', 'high temperature strength']
        
        # 提取性能嵌入
        print("\n  提取性能指标嵌入...")
        for i, property_name in enumerate(properties, 1):
            self.property_embeddings[property_name] = self.get_enhanced_embedding(property_name, 'property')
            if i % 3 == 0 or i == len(properties):
                print(f"    进度: {i}/{len(properties)}")
        
        print("✓ 嵌入向量提取完成")
        
        return self.element_embeddings, self.mechanism_embeddings, self.property_embeddings

class HEAKnowledgeAnalyzer:
    """高熵合金知识分析器"""
    
    def __init__(self, element_embeddings, mechanism_embeddings, property_embeddings):
        self.element_embeddings = element_embeddings
        self.mechanism_embeddings = mechanism_embeddings
        self.property_embeddings = property_embeddings
        
        # 组合所有实体
        self.all_entities = {}
        self.all_entities.update(element_embeddings)
        self.all_entities.update(mechanism_embeddings)
        self.all_entities.update(property_embeddings)
        
        self.entity_names = list(self.all_entities.keys())
        
    def compute_similarity_matrix(self):
        """计算相似度矩阵"""
        print("\n[3/4] 计算相似度矩阵...")
        
        embeddings = np.array([self.all_entities[name] for name in self.entity_names])
        similarity_matrix = cosine_similarity(embeddings)
        similarity_matrix = similarity_matrix.astype(float)
        
        print(f"✓ 相似度矩阵计算完成 (形状: {similarity_matrix.shape})")
        
        return similarity_matrix
    
    def analyze_all_relationships(self, similarity_matrix):
        """分析所有关系"""
        print("\n[4/4] 分析所有实体间关系...")
        
        # 获取索引
        hardness_idx = self.entity_names.index('hardness')
        
        # 1. 元素对硬度的影响
        element_hardness = {}
        for element in self.element_embeddings.keys():
            element_idx = self.entity_names.index(element)
            similarity = similarity_matrix[element_idx][hardness_idx]
            element_hardness[element] = {
                'similarity': float(similarity),
                'atomic_radius': ELEMENT_PROPERTIES.get(element, {}).get('atomic_radius', 0),
                'electronegativity': ELEMENT_PROPERTIES.get(element, {}).get('electronegativity', 0)
            }
        
        # 2. 机制对硬度的影响
        mechanism_hardness = {}
        for mechanism in self.mechanism_embeddings.keys():
            mechanism_idx = self.entity_names.index(mechanism)
            similarity = similarity_matrix[mechanism_idx][hardness_idx]
            mechanism_hardness[mechanism] = float(similarity)
        
        # 3. 元素对机制的影响
        element_mechanism = {}
        for element in self.element_embeddings.keys():
            element_idx = self.entity_names.index(element)
            element_mechanism[element] = {}
            for mechanism in self.mechanism_embeddings.keys():
                mechanism_idx = self.entity_names.index(mechanism)
                similarity = similarity_matrix[element_idx][mechanism_idx]
                element_mechanism[element][mechanism] = float(similarity)
        
        # 4. 元素对其他性能的影响
        element_properties = {}
        for element in self.element_embeddings.keys():
            element_idx = self.entity_names.index(element)
            element_properties[element] = {}
            for property_name in self.property_embeddings.keys():
                if property_name != 'hardness':  # 硬度已经在前面单独处理
                    property_idx = self.entity_names.index(property_name)
                    similarity = similarity_matrix[element_idx][property_idx]
                    element_properties[element][property_name] = float(similarity)
        
        # 5. 机制对其他性能的影响
        mechanism_properties = {}
        for mechanism in self.mechanism_embeddings.keys():
            mechanism_idx = self.entity_names.index(mechanism)
            mechanism_properties[mechanism] = {}
            for property_name in self.property_embeddings.keys():
                if property_name != 'hardness':
                    property_idx = self.entity_names.index(property_name)
                    similarity = similarity_matrix[mechanism_idx][property_idx]
                    mechanism_properties[mechanism][property_name] = float(similarity)
        
        # 6. 所有实体间的完整关系矩阵
        full_relationship_matrix = {}
        for i, entity1 in enumerate(self.entity_names):
            full_relationship_matrix[entity1] = {}
            for j, entity2 in enumerate(self.entity_names):
                if i != j:  # 跳过自相关
                    full_relationship_matrix[entity1][entity2] = float(similarity_matrix[i][j])
        
        print("✓ 关系分析完成")
        
        return {
            'element_hardness': element_hardness,
            'mechanism_hardness': mechanism_hardness,
            'element_mechanism': element_mechanism,
            'element_properties': element_properties,
            'mechanism_properties': mechanism_properties,
            'full_relationship_matrix': full_relationship_matrix,
            'entity_names': self.entity_names
        }

class HEATextReportGenerator:
    """高熵合金文本报告生成器"""
    
    def __init__(self, analysis_results):
        self.results = analysis_results
        
    def generate_detailed_report(self, output_file='HEA_Analysis_Report.txt'):
        """生成详细文本报告"""
        print(f"\n生成详细文本报告: {output_file}")
        
        with open(output_file, 'w', encoding='utf-8') as f:
            # 报告标题
            f.write("=" * 100 + "\n")
            f.write("高熵合金知识图谱深度分析报告\n")
            f.write("=" * 100 + "\n\n")
            
            f.write("报告生成时间: " + datetime.now().strftime('%Y-%m-%d %H:%M:%S') + "\n")
            f.write("分析实体总数: " + str(len(self.results['entity_names'])) + "\n\n")
            
            # 第一部分：元素对硬度影响分析
            self._write_element_hardness_analysis(f)
            
            # 第二部分：强化机制对硬度影响分析
            self._write_mechanism_hardness_analysis(f)
            
            # 第三部分：元素-机制关联矩阵
            self._write_element_mechanism_matrix(f)
            
            # 第四部分：元素对其他性能的影响
            self._write_element_properties_analysis(f)
            
            # 第五部分：机制对其他性能的影响
            self._write_mechanism_properties_analysis(f)
            
            # 第六部分：综合分析与设计建议
            self._write_comprehensive_analysis(f)
            
            # 第七部分：关键发现总结
            self._write_key_findings(f)
        
        print(f"✓ 详细报告已保存到: {output_file}")
        return output_file
    
    def _write_element_hardness_analysis(self, f):
        """写入元素-硬度分析"""
        f.write("第一部分：元素对硬度影响分析\n")
        f.write("-" * 100 + "\n\n")
        
        element_hardness = self.results['element_hardness']
        sorted_elements = sorted(element_hardness.items(), 
                               key=lambda x: x[1]['similarity'], reverse=True)
        
        # 统计信息
        similarities = [data['similarity'] for _, data in sorted_elements]
        mean_sim = np.mean(similarities)
        std_sim = np.std(similarities)
        max_sim = max(similarities)
        min_sim = min(similarities)
        
        f.write("统计特征:\n")
        f.write(f"  • 平均相似度: {mean_sim:.6f}\n")
        f.write(f"  • 标准差: {std_sim:.6f}\n")
        f.write(f"  • 最大值: {max_sim:.6f}\n")
        f.write(f"  • 最小值: {min_sim:.6f}\n")
        f.write(f"  • 极差: {max_sim - min_sim:.6f}\n")
        f.write(f"  • 变异系数: {std_sim/mean_sim:.3f}\n\n")
        
        # 分层阈值
        high_threshold = mean_sim + std_sim
        low_threshold = mean_sim - std_sim
        
        # 详细排名
        f.write("元素硬度关联度详细排名:\n")
        f.write("排名 | 元素 | 相似度 | 原子半径(Å) | 电负性 | 分类\n")
        f.write("-" * 80 + "\n")
        
        for i, (element, data) in enumerate(sorted_elements, 1):
            # 确定元素分类
            element_type = "未知"
            for category, elements in ELEMENT_CATEGORIES.items():
                if element in elements:
                    element_type = category.replace('_', ' ').title()
                    break
            
            # 确定强度标记
            if data['similarity'] > high_threshold:
                strength_marker = "★★★"
            elif data['similarity'] > mean_sim:
                strength_marker = "★★"
            else:
                strength_marker = "★"
            
            f.write(f"{i:4d} | {element:4s} | {data['similarity']:.6f} {strength_marker} | "
                   f"{data['atomic_radius']:11.2f} | {data['electronegativity']:8.2f} | "
                   f"{element_type}\n")
        
        f.write("\n分层分析:\n")
        
        # 高相关层
        f.write(f"【高相关层】(相似度 > {high_threshold:.6f}):\n")
        for element, data in sorted_elements:
            if data['similarity'] > high_threshold:
                f.write(f"  • {element}: {data['similarity']:.6f} "
                       f"(原子半径: {data['atomic_radius']:.2f}Å, "
                       f"电负性: {data['electronegativity']:.2f})\n")
        
        # 中相关层
        f.write(f"\n【中相关层】({low_threshold:.6f} ≤ 相似度 ≤ {high_threshold:.6f}):\n")
        for element, data in sorted_elements:
            if low_threshold <= data['similarity'] <= high_threshold:
                f.write(f"  • {element}: {data['similarity']:.6f} "
                       f"(原子半径: {data['atomic_radius']:.2f}Å, "
                       f"电负性: {data['electronegativity']:.2f})\n")
        
        # 低相关层
        f.write(f"\n【低相关层】(相似度 < {low_threshold:.6f}):\n")
        for element, data in sorted_elements:
            if data['similarity'] < low_threshold:
                f.write(f"  • {element}: {data['similarity']:.6f} "
                       f"(原子半径: {data['atomic_radius']:.2f}Å, "
                       f"电负性: {data['electronegativity']:.2f})\n")
        
        f.write("\n")
    
    def _write_mechanism_hardness_analysis(self, f):
        """写入机制-硬度分析"""
        f.write("第二部分：强化机制对硬度影响分析\n")
        f.write("-" * 100 + "\n\n")
        
        mechanism_hardness = self.results['mechanism_hardness']
        sorted_mechanisms = sorted(mechanism_hardness.items(), 
                                 key=lambda x: x[1], reverse=True)
        
        # 统计信息
        similarities = [sim for _, sim in sorted_mechanisms]
        mean_sim = np.mean(similarities)
        std_sim = np.std(similarities)
        max_sim = max(similarities)
        min_sim = min(similarities)
        
        f.write("统计特征:\n")
        f.write(f"  • 平均相似度: {mean_sim:.6f}\n")
        f.write(f"  • 标准差: {std_sim:.6f}\n")
        f.write(f"  • 最大值: {max_sim:.6f}\n")
        f.write(f"  • 最小值: {min_sim:.6f}\n")
        f.write(f"  • 极差: {max_sim - min_sim:.6f}\n")
        f.write(f"  • 变异系数: {std_sim/mean_sim:.3f}\n\n")
        
        # 分层阈值
        high_threshold = mean_sim + std_sim
        low_threshold = mean_sim - std_sim
        
        # 详细排名
        f.write("强化机制硬度关联度详细排名:\n")
        f.write("排名 | 机制 | 相似度 | 强度等级\n")
        f.write("-" * 80 + "\n")
        
        for i, (mechanism, similarity) in enumerate(sorted_mechanisms, 1):
            # 确定强度标记
            if similarity > high_threshold:
                strength_marker = "★★★"
            elif similarity > mean_sim:
                strength_marker = "★★"
            else:
                strength_marker = "★"
            
            f.write(f"{i:4d} | {mechanism:40s} | {similarity:.6f} | {strength_marker}\n")
        
        f.write("\n机制分类分析:\n")
        
        # 根据机制名称分类
        primary_mechanisms = []
        secondary_mechanisms = []
        plasticity_mechanisms = []
        
        for mechanism, similarity in sorted_mechanisms:
            if 'strengthening' in mechanism:
                primary_mechanisms.append((mechanism, similarity))
            elif 'distortion' in mechanism or 'refinement' in mechanism or 'transformation' in mechanism:
                secondary_mechanisms.append((mechanism, similarity))
            elif 'plasticity' in mechanism:
                plasticity_mechanisms.append((mechanism, similarity))
        
        if primary_mechanisms:
            f.write("【主要强化机制】:\n")
            for mechanism, similarity in primary_mechanisms:
                f.write(f"  • {mechanism}: {similarity:.6f}\n")
        
        if secondary_mechanisms:
            f.write("\n【次要强化机制】:\n")
            for mechanism, similarity in secondary_mechanisms:
                f.write(f"  • {mechanism}: {similarity:.6f}\n")
        
        if plasticity_mechanisms:
            f.write("\n【塑性机制】:\n")
            for mechanism, similarity in plasticity_mechanisms:
                f.write(f"  • {mechanism}: {similarity:.6f}\n")
        
        f.write("\n")
    
    def _write_element_mechanism_matrix(self, f):
        """写入元素-机制关联矩阵"""
        f.write("第三部分：元素-机制关联矩阵\n")
        f.write("-" * 100 + "\n\n")
        
        element_mechanism = self.results['element_mechanism']
        mechanisms = list(next(iter(element_mechanism.values())).keys())
        
        # 按元素对硬度的相似度排序元素
        element_hardness = self.results['element_hardness']
        sorted_elements = sorted(element_hardness.items(), 
                               key=lambda x: x[1]['similarity'], reverse=True)
        elements = [elem for elem, _ in sorted_elements]
        
        # 按机制对硬度的相似度排序机制
        mechanism_hardness = self.results['mechanism_hardness']
        sorted_mech_items = sorted(mechanism_hardness.items(), 
                                 key=lambda x: x[1], reverse=True)
        sorted_mechanisms = [mech for mech, _ in sorted_mech_items]
        
        f.write("关联矩阵 (元素 × 机制):\n\n")
        
        # 表头
        f.write("元素".ljust(8))
        for mechanism in sorted_mechanisms:
            short_name = self._shorten_mechanism_name(mechanism)
            f.write(f" | {short_name:15s}")
        f.write("\n")
        f.write("-" * (8 + len(sorted_mechanisms) * 18) + "\n")
        
        # 矩阵数据
        for element in elements:  # 只显示前15个元素
            f.write(f"{element:8s}")
            for mechanism in sorted_mechanisms:
                similarity = element_mechanism[element][mechanism]
                
                # 根据相似度设置标记
                if similarity > 0.65:
                    marker = "●"
                elif similarity > 0.55:
                    marker = "○"
                else:
                    marker = "·"
                
                f.write(f" | {similarity:.4f}{marker}")
            f.write("\n")
        
        f.write("\n关联强度说明:\n")
        f.write("  ● 强关联 (>0.65)  ○ 中等关联 (0.55-0.65)  · 弱关联 (<0.55)\n\n")
        
        # 找出每个元素最强的关联机制
        f.write("各元素最强关联机制分析:\n")
        for element in elements:
            max_sim = 0
            max_mech = ""
            for mechanism, similarity in element_mechanism[element].items():
                if similarity > max_sim:
                    max_sim = similarity
                    max_mech = mechanism
            
            f.write(f"  • {element}: {self._shorten_mechanism_name(max_mech)} "
                   f"(相似度: {max_sim:.6f})\n")
        
        # 找出每个机制最强的关联元素
        f.write("\n各机制最强关联元素分析:\n")
        for mechanism in sorted_mechanisms[:5]:  # 只显示前5个机制
            max_sim = 0
            max_elem = ""
            for element in elements:
                similarity = element_mechanism[element][mechanism]
                if similarity > max_sim:
                    max_sim = similarity
                    max_elem = element
            
            f.write(f"  • {self._shorten_mechanism_name(mechanism)}: {max_elem} "
                   f"(相似度: {max_sim:.6f})\n")
        
        f.write("\n")
    
    def _write_element_properties_analysis(self, f):
        """写入元素对其他性能的影响"""
        f.write("第四部分：元素对其他性能的影响\n")
        f.write("-" * 100 + "\n\n")
        
        element_properties = self.results['element_properties']
        properties = ['strength', 'ductility', 'toughness', 'wear resistance', 
                     'corrosion resistance', 'high temperature strength']
        
        # 对每个性能，找出关联度最高的元素
        for prop in properties:
            f.write(f"【{prop.upper()}】\n")
            
            # 收集所有元素对该性能的相似度
            prop_similarities = []
            for element in element_properties.keys():
                if prop in element_properties[element]:
                    similarity = element_properties[element][prop]
                    prop_similarities.append((element, similarity))
            
            # 按相似度排序
            prop_similarities.sort(key=lambda x: x[1], reverse=True)
            
            # 显示前5和后3
            f.write("  最佳元素:\n")
            for i in range(min(5, len(prop_similarities))):
                element, similarity = prop_similarities[i]
                f.write(f"    {i+1}. {element}: {similarity:.6f}\n")
            
            f.write("  最差元素:\n")
            for i in range(min(3, len(prop_similarities))):
                idx = len(prop_similarities) - 1 - i
                element, similarity = prop_similarities[idx]
                f.write(f"    {i+1}. {element}: {similarity:.6f}\n")
            
            f.write("\n")
    
    def _write_mechanism_properties_analysis(self, f):
        """写入机制对其他性能的影响"""
        f.write("第五部分：强化机制对其他性能的影响\n")
        f.write("-" * 100 + "\n\n")
        
        mechanism_properties = self.results['mechanism_properties']
        properties = ['strength', 'ductility', 'toughness', 'wear resistance', 
                     'corrosion resistance', 'high temperature strength']
        
        for prop in properties:
            f.write(f"【{prop.upper()}】\n")
            
            # 收集所有机制对该性能的相似度
            prop_similarities = []
            for mechanism in mechanism_properties.keys():
                if prop in mechanism_properties[mechanism]:
                    similarity = mechanism_properties[mechanism][prop]
                    prop_similarities.append((mechanism, similarity))
            
            # 按相似度排序
            prop_similarities.sort(key=lambda x: x[1], reverse=True)
            
            f.write("  机制影响排名:\n")
            for i, (mechanism, similarity) in enumerate(prop_similarities, 1):
                short_name = self._shorten_mechanism_name(mechanism)
                f.write(f"    {i}. {short_name}: {similarity:.6f}\n")
            
            f.write("\n")
    
    def _write_comprehensive_analysis(self, f):
        """写入综合分析"""
        f.write("第六部分：综合分析与设计建议\n")
        f.write("-" * 100 + "\n\n")
        
        element_hardness = self.results['element_hardness']
        mechanism_hardness = self.results['mechanism_hardness']
        element_mechanism = self.results['element_mechanism']
        
        # 元素统计
        elem_sims = [data['similarity'] for data in element_hardness.values()]
        elem_mean = np.mean(elem_sims)
        elem_std = np.std(elem_sims)
        elem_max = max(elem_sims)
        elem_min = min(elem_sims)
        
        # 机制统计
        mech_sims = list(mechanism_hardness.values())
        mech_mean = np.mean(mech_sims)
        mech_std = np.std(mech_sims)
        mech_max = max(mech_sims)
        mech_min = min(mech_sims)
        
        f.write("1. 整体统计分析:\n")
        f.write(f"   元素-硬度关联: 平均值={elem_mean:.6f}, 标准差={elem_std:.6f}, "
               f"范围=[{elem_min:.6f}, {elem_max:.6f}]\n")
        f.write(f"   机制-硬度关联: 平均值={mech_mean:.6f}, 标准差={mech_std:.6f}, "
               f"范围=[{mech_min:.6f}, {mech_max:.6f}]\n\n")
        
        # 寻找最佳元素组合
        f.write("2. 高硬度合金元素组合建议:\n")
        
        # 按类别选择元素
        top_refractory = []
        top_light = []
        top_transition = []
        
        for element, data in sorted(element_hardness.items(), 
                                   key=lambda x: x[1]['similarity'], reverse=True):
            # 分类
            if element in ELEMENT_CATEGORIES['refractory_elements']:
                if len(top_refractory) < 3:
                    top_refractory.append(element)
            elif element in ELEMENT_CATEGORIES['light_elements']:
                if len(top_light) < 2:
                    top_light.append(element)
            elif element in ELEMENT_CATEGORIES['transition_metals']:
                if len(top_transition) < 2:
                    top_transition.append(element)
            
            if len(top_refractory) >= 3 and len(top_light) >= 2 and len(top_transition) >= 2:
                break
        
        # 生成组合建议
        f.write("   a) 以难熔元素为主的体系:\n")
        f.write(f"      推荐: {', '.join(top_refractory[:3])}\n")
        f.write(f"      平均硬度关联度: {np.mean([element_hardness[e]['similarity'] for e in top_refractory[:3]]):.6f}\n")
        
        f.write("\n   b) 难熔+轻元素复合体系:\n")
        f.write(f"      推荐: {', '.join(top_refractory[:2] + top_light[:2])}\n")
        
        f.write("\n   c) 全元素类型平衡体系:\n")
        f.write(f"      推荐: {', '.join(top_refractory[:2] + top_light[:1] + top_transition[:2])}\n")
        
        # 分析最佳机制组合
        f.write("\n3. 强化机制组合策略:\n")
        sorted_mech = sorted(mechanism_hardness.items(), key=lambda x: x[1], reverse=True)
        
        f.write("   a) 主要强化机制 (必须包含):\n")
        for i in range(min(3, len(sorted_mech))):
            mechanism, similarity = sorted_mech[i]
            f.write(f"      • {self._shorten_mechanism_name(mechanism)}: {similarity:.6f}\n")
        
        f.write("\n   b) 辅助强化机制 (选择性包含):\n")
        for i in range(3, min(6, len(sorted_mech))):
            mechanism, similarity = sorted_mech[i]
            f.write(f"      • {self._shorten_mechanism_name(mechanism)}: {similarity:.6f}\n")
        
        f.write("\n4. 元素-机制匹配优化:\n")
        
        # 找出每个高关联元素的顶级机制
        high_elements = []
        for element, data in element_hardness.items():
            if data['similarity'] > elem_mean + elem_std:
                high_elements.append(element)
        
        for element in high_elements[:5]:
            # 找出该元素最强的3个机制
            elem_mech = element_mechanism[element]
            top_mech = sorted(elem_mech.items(), key=lambda x: x[1], reverse=True)[:3]
            
            f.write(f"   • {element} 的最佳激活机制:\n")
            for mechanism, similarity in top_mech:
                f.write(f"      {self._shorten_mechanism_name(mechanism)}: {similarity:.6f}\n")
        
        f.write("\n")
    
    def _write_key_findings(self, f):
        """写入关键发现总结"""
        f.write("第七部分：关键发现总结\n")
        f.write("-" * 100 + "\n\n")
        
        element_hardness = self.results['element_hardness']
        mechanism_hardness = self.results['mechanism_hardness']
        
        # 找出顶级元素
        sorted_elements = sorted(element_hardness.items(), 
                               key=lambda x: x[1]['similarity'], reverse=True)
        top_element = sorted_elements[0]
        bottom_element = sorted_elements[-1]
        
        # 找出顶级机制
        sorted_mechanisms = sorted(mechanism_hardness.items(), 
                                 key=lambda x: x[1], reverse=True)
        top_mechanism = sorted_mechanisms[0]
        bottom_mechanism = sorted_mechanisms[-1]
        
        f.write("1. 最重要的发现:\n")
        f.write(f"   • 对硬度影响最大的元素: {top_element[0]} (相似度: {top_element[1]['similarity']:.6f})\n")
        f.write(f"   • 对硬度影响最小的元素: {bottom_element[0]} (相似度: {bottom_element[1]['similarity']:.6f})\n")
        f.write(f"   • 最有效的强化机制: {self._shorten_mechanism_name(top_mechanism[0])} "
               f"(相似度: {top_mechanism[1]:.6f})\n")
        f.write(f"   • 相对较弱的强化机制: {self._shorten_mechanism_name(bottom_mechanism[0])} "
               f"(相似度: {bottom_mechanism[1]:.6f})\n\n")
        
        # 计算元素和机制的差异
        elem_sims = [data['similarity'] for data in element_hardness.values()]
        mech_sims = list(mechanism_hardness.values())
        
        elem_var_coef = np.std(elem_sims) / np.mean(elem_sims)
        mech_var_coef = np.std(mech_sims) / np.mean(mech_sims)
        
        f.write("2. 差异性分析:\n")
        f.write(f"   • 元素间硬度关联度差异: 变异系数 = {elem_var_coef:.3f}\n")
        f.write(f"   • 机制间硬度关联度差异: 变异系数 = {mech_var_coef:.3f}\n")
        
        if elem_var_coef > mech_var_coef:
            f.write("   • 结论: 元素选择比机制选择对硬度影响更大\n")
        else:
            f.write("   • 结论: 机制选择比元素选择对硬度影响更大\n")
        
        # 物理属性相关性分析
        atomic_radii = []
        electronegativities = []
        similarities = []
        
        for element, data in element_hardness.items():
            atomic_radii.append(data['atomic_radius'])
            electronegativities.append(data['electronegativity'])
            similarities.append(data['similarity'])
        
        # 计算相关性
        corr_atomic = np.corrcoef(atomic_radii, similarities)[0, 1]
        corr_electro = np.corrcoef(electronegativities, similarities)[0, 1]
        
        f.write(f"\n3. 物理属性与硬度关联的相关性:\n")
        f.write(f"   • 原子半径 vs 硬度关联: 相关系数 = {corr_atomic:.3f}\n")
        f.write(f"   • 电负性 vs 硬度关联: 相关系数 = {corr_electro:.3f}\n")
        
        if abs(corr_atomic) > 0.3:
            if corr_atomic > 0:
                f.write("   • 原子半径越大，硬度关联度越高\n")
            else:
                f.write("   • 原子半径越小，硬度关联度越高\n")
        
        if abs(corr_electro) > 0.3:
            if corr_electro > 0:
                f.write("   • 电负性越大，硬度关联度越高\n")
            else:
                f.write("   • 电负性越小，硬度关联度越高\n")
        
        f.write("\n4. 合金设计核心原则:\n")
        f.write("   • 优先选择高硬度关联度的难熔元素 (W, Ta, Nb, Mo)\n")
        f.write("   • 结合多种强化机制，特别是固溶强化和析出强化\n")
        f.write("   • 考虑元素间的协同效应，避免元素间负面相互作用\n")
        f.write("   • 在追求高硬度的同时，兼顾其他性能如韧性和耐腐蚀性\n")
        
        f.write("\n" + "=" * 100 + "\n")
        f.write("报告结束\n")
        f.write("=" * 100 + "\n")
    
    def _shorten_mechanism_name(self, mechanism_name):
        """缩短机制名称以便显示"""
        words = mechanism_name.split()
        if len(words) > 3:
            return ' '.join(words[:3])
        return mechanism_name

# ============================================================================
# 主程序
# ============================================================================

def main():
    """主函数"""
    
    # 1. 初始化提取器
    extractor = HEAKnowledgeExtractor(MODEL_PATH)
    
    # 2. 提取嵌入向量
    element_embeddings, mechanism_embeddings, property_embeddings = extractor.extract_all_embeddings()
    
    # 3. 初始化分析器
    analyzer = HEAKnowledgeAnalyzer(element_embeddings, mechanism_embeddings, property_embeddings)
    
    # 4. 计算相似度矩阵
    similarity_matrix = analyzer.compute_similarity_matrix()
    
    # 5. 分析所有关系
    relationships = analyzer.analyze_all_relationships(similarity_matrix)
    
    # 6. 生成详细文本报告
    report_generator = HEATextReportGenerator(relationships)
    report_file = report_generator.generate_detailed_report()
    
    # 7. 在控制台输出关键摘要
    print("\n" + "=" * 80)
    print("关键摘要")
    print("=" * 80)
    
    element_hardness = relationships['element_hardness']
    mechanism_hardness = relationships['mechanism_hardness']
    
    # 找出顶级元素
    sorted_elements = sorted(element_hardness.items(), 
                           key=lambda x: x[1]['similarity'], reverse=True)
    
    print("\nTop 5 元素 (硬度关联度最高):")
    for i, (element, data) in enumerate(sorted_elements[:5], 1):
        print(f"  {i}. {element}: {data['similarity']:.6f}")
    
    print("\nBottom 5 元素 (硬度关联度最低):")
    for i, (element, data) in enumerate(sorted_elements[-5:], 1):
        print(f"  {i}. {element}: {data['similarity']:.6f}")
    
    # 找出顶级机制
    sorted_mechanisms = sorted(mechanism_hardness.items(), 
                             key=lambda x: x[1], reverse=True)
    
    print("\nTop 3 强化机制:")
    for i, (mechanism, similarity) in enumerate(sorted_mechanisms[:3], 1):
        print(f"  {i}. {mechanism[:30]}...: {similarity:.6f}")
    
    # 计算统计
    elem_sims = [data['similarity'] for data in element_hardness.values()]
    mech_sims = list(mechanism_hardness.values())
    
    print(f"\n统计摘要:")
    print(f"  元素平均关联度: {np.mean(elem_sims):.6f}")
    print(f"  元素关联度范围: {np.min(elem_sims):.6f} - {np.max(elem_sims):.6f}")
    print(f"  机制平均关联度: {np.mean(mech_sims):.6f}")
    print(f"  机制关联度范围: {np.min(mech_sims):.6f} - {np.max(mech_sims):.6f}")
    
    # 元素-机制关联示例
    print("\n元素-机制强关联示例:")
    element_mechanism = relationships['element_mechanism']
    
    # 选择几个元素显示
    example_elements = ['W', 'Ta', 'Nb', 'Mo', 'Al']
    for element in example_elements:
        if element in element_mechanism:
            # 找出该元素最强的机制
            top_mech = max(element_mechanism[element].items(), key=lambda x: x[1])
            print(f"  {element} -> {top_mech[0][:25]}...: {top_mech[1]:.6f}")
    
    print("\n" + "=" * 80)
    print(f"详细报告已保存到: {report_file}")
    print("报告包含:")
    print("  • 所有元素与硬度的详细关联度")
    print("  • 所有机制与硬度的详细关联度")
    print("  • 元素-机制完整关联矩阵")
    print("  • 元素对其他性能的影响")
    print("  • 机制对其他性能的影响")
    print("  • 综合分析与设计建议")
    print("=" * 80)

if __name__ == "__main__":
    main()

高熵合金知识图谱系统 - 纯文本报告版
[1/4] 加载预训练模型...

[2/4] 提取实体嵌入向量...
  提取元素嵌入...
    进度: 5/20
    进度: 10/20
    进度: 15/20
    进度: 20/20

  提取强化机制嵌入...
    进度: 2/8
    进度: 4/8
    进度: 6/8
    进度: 8/8

  提取性能指标嵌入...
    进度: 3/7
    进度: 6/7
    进度: 7/7
✓ 嵌入向量提取完成

[3/4] 计算相似度矩阵...
✓ 相似度矩阵计算完成 (形状: (35, 35))

[4/4] 分析所有实体间关系...
✓ 关系分析完成

生成详细文本报告: HEA_Analysis_Report.txt
✓ 详细报告已保存到: HEA_Analysis_Report.txt

关键摘要

Top 5 元素 (硬度关联度最高):
  1. W: 0.762849
  2. Mg: 0.761194
  3. Al: 0.753390
  4. V: 0.750899
  5. Ta: 0.746429

Bottom 5 元素 (硬度关联度最低):
  1. Cr: 0.731596
  2. Mo: 0.730393
  3. Sn: 0.729986
  4. Co: 0.726694
  5. Si: 0.715087

Top 3 强化机制:
  1. phase transformation...: 0.660196
  2. grain refinement...: 0.637104
  3. lattice distortion...: 0.613874

统计摘要:
  元素平均关联度: 0.739699
  元素关联度范围: 0.715087 - 0.762849
  机制平均关联度: 0.605610
  机制关联度范围: 0.568017 - 0.660196

元素-机制强关联示例:
  W -> lattice distortion...: 0.691584
  Ta -> precipitation strengtheni...: 0.670772
  Nb -> precipitation strengtheni...: 0.694829

In [18]:
import os

output_dir = "outputs"

# 检查是否为目录（更准确，避免与文件混淆）
if not os.path.isdir(output_dir):
    os.makedirs(output_dir, exist_ok=True)
    print(f"已创建目录: {output_dir}")
else:
    print(f"目录已存在: {output_dir}")

目录已存在: outputs


In [20]:
import matplotlib.pyplot as plt
import numpy as np
from matplotlib.patches import Circle
import re

# 禁用中文字体
plt.rcParams['font.sans-serif'] = ['DejaVu Sans']
plt.rcParams['axes.unicode_minus'] = False

# ============================================================================
# 从报告文件中提取数据（改进版）
# ============================================================================
def extract_data_from_report(file_path):
    with open(file_path, 'r', encoding='utf-8') as f:
        lines = f.readlines()
    
    elements_data = {}
    mechanisms_data = {}
    element_mechanism_matrix = {}
    
    current_section = None
    
    for line in lines:
        line = line.strip()
        
        # 识别当前部分
        if '元素硬度关联度详细排名:' in line:
            current_section = 'elements'
            continue
        elif '强化机制硬度关联度详细排名:' in line:
            current_section = 'mechanisms'
            continue
        elif '关联矩阵 (元素 × 机制):' in line:
            current_section = 'matrix'
            continue
        
        # 处理元素数据
        if current_section == 'elements':
            # 匹配类似 "1 | W    | 0.762849 ★★★ |        1.39 |     2.36 | Refractory Elements"
            match = re.match(r'\s*(\d+)\s*\|\s*([A-Za-z]+)\s*\|\s*([\d.]+)', line)
            if match:
                element = match.group(2)
                similarity = float(match.group(3))
                elements_data[element] = similarity
        
        # 处理机制数据
        elif current_section == 'mechanisms':
            # 匹配类似 "1 | phase transformation                     | 0.660196 | ★★★"
            match = re.match(r'\s*(\d+)\s*\|\s*([a-zA-Z\s]+)\s*\|\s*([\d.]+)', line)
            if match:
                mechanism = match.group(2).strip()
                similarity = float(match.group(3))
                mechanisms_data[mechanism] = similarity
        
        # 处理关联矩阵数据
        elif current_section == 'matrix':
            # 跳过表头行
            if '|' in line and 'phase transform' not in line and '元素' not in line:
                # 分割行，获取各个字段
                parts = [p.strip() for p in line.split('|')]
                if len(parts) >= 9:  # 确保有足够的列
                    element = parts[0]
                    if element and element in elements_data:  # 只处理已存在的元素
                        # 创建简化的机制关联数据
                        mechanisms = {}
                        
                        # 机制名称映射
                        mech_names = [
                            'phase transformation',
                            'grain refinement',
                            'lattice distortion',
                            'precipitation strengthening',
                            'dislocation strengthening',
                            'transformation induced plasticity',
                            'solid solution strengthening',
                            'twining induced plasticity'
                        ]
                        
                        for i in range(1, 9):
                            if i < len(parts):
                                val_str = parts[i]
                                # 提取数字
                                num_match = re.search(r'[\d.]+', val_str)
                                if num_match:
                                    mechanisms[mech_names[i-1]] = float(num_match.group())
                        
                        if mechanisms:
                            element_mechanism_matrix[element] = mechanisms
    
    return elements_data, mechanisms_data, element_mechanism_matrix

# ============================================================================
# 图1: 元素-硬度放射状图
# ============================================================================
def create_element_hardness_graph(elements_data):
    if not elements_data:
        print("警告: 没有提取到元素数据，使用示例数据")
        # 使用示例数据
        elements_data = {
            'W': 0.7628, 'Mg': 0.7612, 'Al': 0.7534, 'V': 0.7509, 'Ta': 0.7464,
            'Nb': 0.7444, 'Hf': 0.7426, 'Ti': 0.7418, 'Zr': 0.7410, 'Cu': 0.7381,
            'Ni': 0.7371, 'Fe': 0.7363, 'Zn': 0.7352, 'Sc': 0.7347, 'Mn': 0.7343,
            'Cr': 0.7316, 'Mo': 0.7304, 'Sn': 0.7300, 'Co': 0.7267, 'Si': 0.7151
        }
    
    fig, ax = plt.subplots(figsize=(20, 20), facecolor='white')
    
    # 按相似度排序
    sorted_elements = sorted(elements_data.items(), key=lambda x: x[1], reverse=True)
    n = len(sorted_elements)
    angles = np.linspace(0, 2*np.pi, n, endpoint=False)
    
    center_x, center_y = 0, 0
    
    # 获取相似度的最小值和最大值用于缩放
    similarities = [sim for _, sim in sorted_elements]
    min_sim = min(similarities)
    max_sim = max(similarities)
    
    def scale_radius(similarity):
        norm = (similarity - min_sim) / (max_sim - min_sim)
        scaled = norm ** 2.5
        return 3 + (1 - scaled) * 5.5  # 反向：相似度高=距离近
    
    # 绘制背景同心圆
    for r in [2, 4, 6, 8, 10]:
        circle = Circle((center_x, center_y), r, fill=False, 
                       edgecolor='#E0E0E0', linewidth=1, 
                       linestyle='--', alpha=0.3, zorder=0)
        ax.add_patch(circle)
    
    # 中心节点 - HV (更大的圆圈)
    center_size = 1.5  # 增大中心圆圈大小
    center_circle = Circle((center_x, center_y), center_size, 
                          color='#FF4757', ec='#8B0000', 
                          linewidth=8, zorder=100, alpha=0.95)
    ax.add_patch(center_circle)
    ax.text(center_x, center_y, 'HV', ha='center', va='center',
            fontsize=60, fontweight='bold', color='white', 
            zorder=101, family='serif')
    
    # 颜色方案 - 渐变
    def get_element_color(similarity):
        norm = (similarity - min_sim) / (max_sim - min_sim)
        # 从深红到浅橙
        r = 0.8 + norm * 0.2
        g = 0.1 + norm * 0.5
        b = 0.1
        return (r, g, b)
    
    # 绘制元素
    for idx, ((elem, sim), angle) in enumerate(zip(sorted_elements, angles)):
        radius = scale_radius(sim)
        x = center_x + radius * np.cos(angle)
        y = center_y + radius * np.sin(angle)
        
        # 连线粗细
        norm = (sim - min_sim) / (max_sim - min_sim)
        line_width = 4 + norm * 15
        
        color = get_element_color(sim)
        
        # 绘制连线
        ax.plot([center_x, x], [center_y, y], 
               color=color, linewidth=line_width, 
               alpha=0.7, zorder=5, solid_capstyle='round')
        
        # 元素节点
        node_size = 0.5 + norm * 0.3
        elem_circle = Circle((x, y), node_size, color=color, 
                            ec='#2C3E50', linewidth=4, 
                            zorder=15, alpha=0.9)
        ax.add_patch(elem_circle)
        
        # 元素标签（只在圈外显示）
        label_distance = radius + 2.0  # 增加距离
        label_x = center_x + label_distance * np.cos(angle)
        label_y = center_y + label_distance * np.sin(angle)
        
        # 根据位置调整对齐
        if -np.pi/4 < angle < np.pi/4 or angle > 7*np.pi/4:
            ha = 'left'
        elif 3*np.pi/4 < angle < 5*np.pi/4:
            ha = 'right'
        else:
            ha = 'center'
        
        ax.text(label_x, label_y, elem, 
               ha=ha, va='center', fontsize=26,  # 增大字体
               fontweight='bold', color='#2C3E50',
               bbox=dict(boxstyle='round,pad=0.5', 
                        facecolor='white', 
                        edgecolor=color, linewidth=4, 
                        alpha=0.9))
    
    ax.set_xlim(-13, 13)
    ax.set_ylim(-13, 13)
    ax.set_aspect('equal')
    ax.axis('off')
    
    plt.tight_layout()
    plt.savefig('outputs/hea_element_hardness.png', 
                dpi=600, bbox_inches='tight', facecolor='white')
    plt.close()
    print("✓ 图1: 元素-硬度关联图已生成")

# ============================================================================
# 图2: 强化机制-硬度放射图
# ============================================================================
def create_mechanism_hardness_graph(mechanisms_data):
    if not mechanisms_data:
        print("警告: 没有提取到机制数据，使用示例数据")
        # 使用示例数据
        mechanisms_data = {
            'phase transformation': 0.6602,
            'grain refinement': 0.6371,
            'lattice distortion': 0.6139,
            'precipitation strengthening': 0.6108,
            'dislocation strengthening': 0.5916,
            'transformation induced plasticity': 0.5911,
            'solid solution strengthening': 0.5722,
            'twining induced plasticity': 0.5680
        }
    
    # 机制名称映射到简短版本
    short_names = {
            'phase transformation': 'Phase\nTransformation',
            'grain refinement': 'Grain\nRefinement',
            'lattice distortion': 'Lattice\nDistortion',
            'precipitation strengthening': 'Precipitation\nStrengthening',
            'dislocation strengthening': 'Dislocation\nstrengthening',
            'transformation induced plasticity': 'Transformation\nInduced\nPlasticity',
            'solid solution strengthening': 'Solid\nSolution\nStrengthening',
            'twining induced plasticity': 'Twining\nInduced\nPlasticity'
    }
    
    fig, ax = plt.subplots(figsize=(18, 18), facecolor='white')
    
    # 按相似度排序
    sorted_mechanisms = sorted(mechanisms_data.items(), key=lambda x: x[1], reverse=True)
    n = len(sorted_mechanisms)
    angles = np.linspace(0, 2*np.pi, n, endpoint=False)
    
    center_x, center_y = 0, 0
    
    min_sim = min(mechanisms_data.values())
    max_sim = max(mechanisms_data.values())
    
    def scale_radius(similarity):
        norm = (max_sim - similarity) / (max_sim - min_sim)
        return 2.5 + norm * 5
    
    # 计算最大标签距离，用于确定坐标轴范围
    max_radius = 0
    label_distances = {}
    
    for mech, sim in sorted_mechanisms:
        radius = scale_radius(sim)
        max_radius = max(max_radius, radius)
        
        # 为每个机制计算标签距离
        if mech == "phase transformation":
            label_distances[mech] = radius + 4.5
        elif mech == "grain refinement":
            label_distances[mech] = radius + 3.5
        elif mech == "lattice distortion":
            label_distances[mech] = radius + 2.5
        elif mech == "precipitation strengthening":
            label_distances[mech] = radius + 3.5
        elif mech == "dislocation strengthening":
            label_distances[mech] = radius + 4.0
        elif mech == "transformation induced plasticity":
            label_distances[mech] = radius + 3.5
        elif mech == "solid solution strengthening":
            label_distances[mech] = radius + 2.5
        elif mech == "twining induced plasticity":
            label_distances[mech] = radius + 3.5
        else:
            label_distances[mech] = radius + 3.5
        
        max_radius = max(max_radius, label_distances[mech])
    
    # 设置坐标轴范围，确保所有内容都可见
    # 考虑背景圆的最大半径(8)和最大标签距离
    plot_margin = 2.0  # 额外的边距
    plot_range = max(max_radius, 8) + plot_margin
    ax.set_xlim(-plot_range, plot_range)
    ax.set_ylim(-plot_range, plot_range)
    
    # 背景圆
    for r in [2, 4, 6, 8]:
        circle = Circle((center_x, center_y), r, fill=False, 
                       edgecolor='#D0D0D0', linewidth=1.5, 
                       linestyle=':', alpha=0.3, zorder=0)
        ax.add_patch(circle)
    
    # 中心 - HV (更大的圆圈)
    center_size = 1.5
    center_circle = Circle((center_x, center_y), center_size, 
                          color='#5F27CD', ec='#341F97', 
                          linewidth=8, zorder=100, alpha=0.95)
    ax.add_patch(center_circle)
    ax.text(center_x, center_y, 'HV', ha='center', va='center',
            fontsize=60, fontweight='bold', color='white', 
            zorder=101, family='serif')
    
    # 彩虹色谱
    colors = plt.cm.viridis(np.linspace(0.2, 0.95, n))
    
    for idx, ((mech, sim), angle, color) in enumerate(zip(sorted_mechanisms, angles, colors)):
        radius = scale_radius(sim)
        x = center_x + radius * np.cos(angle)
        y = center_y + radius * np.sin(angle)
        
        norm = (sim - min_sim) / (max_sim - min_sim)
        line_width = 5 + norm * 16
        
        # 连线
        ax.plot([center_x, x], [center_y, y], 
               color=color, linewidth=line_width, 
               alpha=0.8, zorder=5, solid_capstyle='round')
        
        # 节点
        node_size = 0.6 + norm * 0.35
        mech_circle = Circle((x, y), node_size, color=color, 
                            ec='black', linewidth=4.5, zorder=15, alpha=0.9)
        ax.add_patch(mech_circle)
        
        # 标签
        label_distance = label_distances[mech]
        label_x = center_x + label_distance * np.cos(angle)
        label_y = center_y + label_distance * np.sin(angle)
        
        short_name = short_names.get(mech, mech)
        ax.text(label_x, label_y, short_name, 
               ha='center', va='center', fontsize=28,
               fontweight='bold', color='#1A1A1A',
               bbox=dict(boxstyle='round,pad=1.2',
                        facecolor='white', 
                        edgecolor=color, linewidth=3, alpha=0.95))
    
    ax.set_aspect('equal')
    ax.axis('off')
    
    # 调试信息，查看实际范围
    print(f"设置的坐标轴范围: x=[{-plot_range:.2f}, {plot_range:.2f}], y=[{-plot_range:.2f}, {plot_range:.2f}]")
    print(f"最大半径: {max_radius:.2f}, 背景圆最大半径: 8.0, 边距: {plot_margin}")
    
    plt.tight_layout(pad=2.0)  # 增加tight_layout的边距
    plt.savefig('outputs/hea_mechanism_hardness.png', 
                dpi=600, facecolor='white', bbox_inches='tight', pad_inches=0.3)
    plt.close()
    print("✓ 图2: 强化机制-硬度关联图已生成")

# ============================================================================
# 图3: 元素-机制关联网络图
# ============================================================================
def create_element_mechanism_network(element_mechanism_matrix, elements_data):
    if not element_mechanism_matrix:
        print("警告: 没有提取到关联矩阵数据，使用示例数据")
        # 创建示例数据
        element_mechanism_matrix = {}
        mechanisms = ['phase transformation', 'grain refinement', 'lattice distortion', 
                     'precipitation strengthening', 'dislocation strengthening',
                     'transformation induced plasticity', 'solid solution strengthening',
                     'twining induced plasticity']
        
        # 为每个元素创建一些示例关联数据
        for element in elements_data.keys():
            correlations = {}
            for mech in mechanisms:
                # 生成基于元素硬度的随机关联度
                base_corr = elements_data[element] * 0.85 + np.random.uniform(-0.05, 0.05)
                correlations[mech] = min(0.75, max(0.45, base_corr))
            element_mechanism_matrix[element] = correlations
    
    # 只使用前15个元素，避免过于拥挤
    available_elements = list(elements_data.keys())#[:15]
    
    # 机制名称映射
    mechanism_names = {
            'phase transformation': 'Phase\nTransformation',
            'grain refinement': 'Grain\nRefinement',
            'lattice distortion': 'Lattice\nDistortion',
            'precipitation strengthening': 'Precipitation\nStrengthening',
            'dislocation strengthening': 'Dislocation\nstrengthening',
            'transformation induced plasticity': 'Transformation\nInduced\nPlasticity',
            'solid solution strengthening': 'Solid\nSolution\nStrengthening',
            'twining induced plasticity': 'Twining\nInduced\nPlasticity'
    }
    
    fig, ax = plt.subplots(figsize=(22, 22), facecolor='white')
    
    n_elements = len(available_elements)
    n_mechanisms = len(mechanism_names)
    
    # 元素节点角度（外圈）
    element_angles = np.linspace(0, 2*np.pi, n_elements, endpoint=False)
    
    # 机制节点角度（内圈）
    mechanism_angles = np.linspace(0, 2*np.pi, n_mechanisms, endpoint=False)
    
    center_x, center_y = 0, 0
    
    # 绘制背景圆
    for r in [3, 6, 9]:
        circle = Circle((center_x, center_y), r, fill=False, 
                       edgecolor='#E0E0E0', linewidth=1.5, 
                       linestyle='--', alpha=0.3, zorder=0)
        ax.add_patch(circle)
    
    # 绘制元素节点（外圈）- 使用统一颜色
    element_radius = 9
    element_color = '#E74C3C'  # 统一的红色

    for idx, (element, angle) in enumerate(zip(available_elements, element_angles)):
        x = center_x + element_radius * np.cos(angle)
        y = center_y + element_radius * np.sin(angle)
        
        # 元素节点 - 使用统一颜色
        elem_circle = Circle((x, y), 0.7, color=element_color, 
                            ec='#2C3E50', linewidth=3.5, zorder=20, alpha=0.9)
        ax.add_patch(elem_circle)
        
        # 元素标签
        label_distance = element_radius + 1.8
        label_x = center_x + label_distance * np.cos(angle)
        label_y = center_y + label_distance * np.sin(angle)
        
        # 对齐方式
        if -np.pi/4 < angle < np.pi/4 or angle > 7*np.pi/4:
            ha = 'left'
        elif 3*np.pi/4 < angle < 5*np.pi/4:
            ha = 'right'
        else:
            ha = 'center'
        
        ax.text(label_x, label_y, element, 
            ha=ha, va='center', fontsize=30, fontweight='bold',
            color='#2C3E50',
            bbox=dict(boxstyle='round,pad=0.8', 
                        facecolor='white', 
                        edgecolor=element_color,  # 使用统一颜色
                        linewidth=3, alpha=0.9))

    # 绘制机制节点（内圈）- 使用统一颜色
    mechanism_radius = 4.5
    mechanism_color = '#3498DB'  # 统一的蓝色

    mechanism_list = list(mechanism_names.keys())

    for idx, (mechanism, angle) in enumerate(zip(mechanism_list, mechanism_angles)):
        x = center_x + mechanism_radius * np.cos(angle)
        y = center_y + mechanism_radius * np.sin(angle)
        
        # 机制节点 - 使用统一颜色
        mech_circle = Circle((x, y), 0.6, color=mechanism_color, 
                            ec='black', linewidth=3.5, zorder=20, alpha=0.9)
        ax.add_patch(mech_circle)
        
        # 机制标签 - 使用简短名称，加大字体
        label_distance = mechanism_radius + 2.2
        label_x = center_x + label_distance * np.cos(angle)
        label_y = center_y + label_distance * np.sin(angle)
        
        short_name = mechanism_names[mechanism]
        ax.text(label_x, label_y, short_name, 
            ha='center', va='center', fontsize=20, fontweight='bold',
            color='#1A1A1A',
            bbox=dict(boxstyle='round,pad=0.8', 
                        facecolor='white', 
                        edgecolor=mechanism_color,  # 使用统一颜色
                        linewidth=3, alpha=0.9))
    
    # 收集所有关联度值，用于归一化
    all_correlations = []
    for element in available_elements:
        if element in element_mechanism_matrix:
            for mechanism in mechanism_list:
                if mechanism in element_mechanism_matrix[element]:
                    all_correlations.append(element_mechanism_matrix[element][mechanism])
    
    if all_correlations:
        min_corr = min(all_correlations)
        max_corr = max(all_correlations)
        corr_range = max_corr - min_corr if max_corr > min_corr else 1
    else:
        min_corr, max_corr, corr_range = 0, 1, 1
    
    # 绘制元素-机制连线 - 改进的视觉映射
    # 绘制元素-机制连线 - 优化的视觉映射
    for elem_idx, element in enumerate(available_elements):
        elem_angle = element_angles[elem_idx]
        elem_x = center_x + element_radius * np.cos(elem_angle)
        elem_y = center_y + element_radius * np.sin(elem_angle)
        
        if element in element_mechanism_matrix:
            for mech_idx, mechanism in enumerate(mechanism_list):
                if mechanism in element_mechanism_matrix[element]:
                    correlation = element_mechanism_matrix[element][mechanism]
                    
                    mech_angle = mechanism_angles[mech_idx]
                    mech_x = center_x + mechanism_radius * np.cos(mech_angle)
                    mech_y = center_y + mechanism_radius * np.sin(mech_angle)
                    
                    # 归一化关联度到 0-1 范围
                    norm_corr = (correlation - min_corr) / corr_range
                    







                    # 线宽：更温和的范围，从0.3到4.5
                    line_width = 0.3 + norm_corr * 4.2  # 从0.3到4.5
                    
                    # 透明度：更温和的映射，增强区分度
                    line_alpha = 0.15 + norm_corr ** 2 * 0.6  # 从0.15到0.75
                    











                
                    # 颜色：使用渐变色图（反转）
                    cmap = plt.cm.RdYlGn  
                    line_color = cmap(1 - norm_corr)
                    
                    # 绘制连线
                    ax.plot([elem_x, mech_x], [elem_y, mech_y], 
                        color=line_color, linewidth=line_width, 
                        alpha=line_alpha, zorder=10)
    
    # 添加颜色条图例
    from matplotlib.colorbar import ColorbarBase
    from matplotlib.colors import Normalize
    
    # 在右侧添加颜色条
    # cax = fig.add_axes([0.92, 0.3, 0.02, 0.4])
    # norm = Normalize(vmin=min_corr, vmax=max_corr)
    # cb = ColorbarBase(cax, cmap=plt.cm.RdYlGn, norm=norm, orientation='vertical')
    # cb.set_label('Correlation Strength', fontsize=18, fontweight='bold')
    # cb.ax.tick_params(labelsize=14)
    
    ax.set_xlim(-12, 12)
    ax.set_ylim(-12, 12)
    ax.set_aspect('equal')
    ax.axis('off')
    
    plt.tight_layout()
    plt.savefig('outputs/hea_element_mechanism_network.png', 
                dpi=600, bbox_inches='tight', facecolor='white')
    plt.close()
    print("✓ 图3: 元素-机制关联网络图已生成")

# ============================================================================
# 主执行
# ============================================================================
if __name__ == '__main__':
    print("\n" + "="*60)
    print("开始从报告文件提取数据并生成HEA知识图谱可视化")
    print("="*60 + "\n")
    
    # 从报告中提取数据
    report_file = 'HEA_Analysis_Report.txt'
    elements_data, mechanisms_data, element_mechanism_matrix = extract_data_from_report(report_file)
    
    print(f"提取到 {len(elements_data)} 个元素数据")
    print(f"提取到 {len(mechanisms_data)} 个强化机制数据")
    print(f"提取到 {len(element_mechanism_matrix)} 个元素的关联矩阵数据")
    
    # 生成可视化图表
    create_element_hardness_graph(elements_data)
    create_mechanism_hardness_graph(mechanisms_data)
    create_element_mechanism_network(element_mechanism_matrix, elements_data)
    
    print("\n" + "="*60)
    print("✅ 所有可视化图表生成完成!")
    print("="*60 + "\n")
    print("生成的图片:")
    print("1. hea_element_hardness.png - 元素-硬度关联图")
    print("2. hea_mechanism_hardness.png - 机制-硬度关联图")
    print("3. hea_element_mechanism_network.png - 元素-机制关联网络图")


开始从报告文件提取数据并生成HEA知识图谱可视化

提取到 20 个元素数据
提取到 8 个强化机制数据
提取到 20 个元素的关联矩阵数据
✓ 图1: 元素-硬度关联图已生成
设置的坐标轴范围: x=[-13.00, 13.00], y=[-13.00, 13.00]
最大半径: 11.00, 背景圆最大半径: 8.0, 边距: 2.0
✓ 图2: 强化机制-硬度关联图已生成
✓ 图3: 元素-机制关联网络图已生成

✅ 所有可视化图表生成完成!

生成的图片:
1. hea_element_hardness.png - 元素-硬度关联图
2. hea_mechanism_hardness.png - 机制-硬度关联图
3. hea_element_mechanism_network.png - 元素-机制关联网络图
